# The model, one stage at a time

The model is TabICL v2 with photos in its rows: a column stage, a row stage and an in-context stage, pretrained on synthetic tables, plus an image side that turns each row's photo into tokens that join the row stage next to the row's cells. This notebook takes one small table through it. Every section prints the code of the stage as it is in `table_image_icl/` and then runs it, so the shapes and a few values can be checked against the prose. The table stages carry TabICL's released weights; the image side is untrained and tiny, since what matters here is what each stage is allowed to see, not what it predicts.

In [1]:
import os, sys, inspect
os.chdir(next(p for p in [os.getcwd(), os.path.dirname(os.getcwd())] if os.path.exists(os.path.join(p, "table_image_icl"))))
sys.path.insert(0, os.getcwd())
import torch
torch.manual_seed(0); torch.set_printoptions(precision=2, sci_mode=False, linewidth=120)

def show(*objects):
    for o in objects:
        print(inspect.getsource(o))

def changed(a, b, dims=(-1,)):
    """1 where a differs from b by more than float noise, after collapsing the given dims"""
    d = (a - b).abs()
    for dim in sorted((x % a.dim() for x in dims), reverse=True):
        d = d.amax(dim)
    return (d > 1e-4).int()

## 1. The inputs

One task is a table whose first rows are the context (values and label known) and whose last rows are the test rows (values known, label to predict). Here: 3 columns, 5 context rows, 3 test rows, one task in the batch. The first two columns are the same numbers in different units, the third has a missing value.

Labels are label slots, not class ids: a task's classes sit on any of 10 slots, so the model cannot learn that slot 0 means anything in particular. This task uses slots 3 and 5. The photos are 28 px, so encoder `pixels` with 14 px patches gives a 2 x 2 grid of 4 patches per photo.

In [2]:
from table_image_icl.model import TableImageICL, ModelConfig

nan = float("nan")
x = torch.tensor([[[1.0, 10.0, 0.5],      # context rows: values and label known
                   [2.0, 20.0, 0.1],
                   [3.0, 30.0, nan],
                   [4.0, 40.0, 0.9],
                   [5.0, 50.0, 0.3],
                   [1.5, 15.0, 0.4],      # test rows: label to predict
                   [4.5, 45.0, 0.2],
                   [6.0, 60.0, 0.8]]])
y_context = torch.tensor([[3, 5, 3, 5, 3]])
y_test = torch.tensor([[3, 5, 5]])
n_context = y_context.shape[1]
images = torch.randn(1, x.shape[1], 3, 28, 28)                        # one random 28 px photo per row

cfg = ModelConfig(encoder="pixels", image_size=28, encoder_layers=1, patch_layers=1)
model = TableImageICL(cfg).eval()                                         # TabICL's weights, a fresh image side
print("x        ", tuple(x.shape), "(batch, rows, columns)")
print("y_context", tuple(y_context.shape), "label slots of the context rows:", y_context[0].tolist())
print("images   ", tuple(images.shape), "(batch, rows, channels, height, width)")
for name, module in model.named_children():
    print(f"{name:13s} {sum(p.numel() for p in module.parameters()) / 1e6:6.2f}M parameters", "(TabICL v2, pretrained)" if name == "table" else "(ours)")

x         (1, 8, 3) (batch, rows, columns)
y_context (1, 5) label slots of the context rows: [3, 5, 3, 5, 3]
images    (1, 8, 3, 28, 28) (batch, rows, channels, height, width)
table          27.55M parameters (TabICL v2, pretrained)
embed           0.08M parameters (ours)
encoder         0.13M parameters (ours)
read            0.13M parameters (ours)


## 2. The column stage

TabICL first standardises every column with the mean and spread of its context rows, fills missing cells with the mean, and softens outliers, so units do not matter and a test row cannot shift the scale. Then each column is handled on its own: its cells attend to each other through a set of inducing tokens, with the context cells' labels added, so a cell comes out knowing where it sits in its column's distribution and how that relates to the label. The output is one vector per cell, plus four CLS tokens per row that the row stage will pool. TabICL wants classes numbered 0..k-1, so the slots are renumbered first.

In [3]:
from table_image_icl.model import prepare_table, contiguous_codes

show(prepare_table, contiguous_codes)

def prepare_table(x: Tensor, n_context: int) -> Tensor:
    """TabICL's preprocessing, fitted on the context rows of each table (batch, rows, columns): standardise, fill
    missing cells with the mean, clip, then soften values beyond OUTLIER_SIGMAS of the cleaned context with a log."""
    context = x[:, :n_context]
    valid = ~torch.isnan(context)
    count = valid.sum(1, keepdim=True).clamp_min(1)
    mean = torch.where(valid, context, 0.0).sum(1, keepdim=True) / count
    std = (torch.where(valid, (context - mean) ** 2, 0.0).sum(1, keepdim=True) / count).sqrt() + 1e-6
    z = ((x - mean) / std).nan_to_num(0.0).clamp(-100, 100)
    lower, upper = outlier_bounds(z[:, :n_context])
    z = torch.maximum(-torch.log1p(z.abs()) + lower, z)
    return torch.minimum(torch.log1p(z.abs()) + upper, z)

def contiguous_codes(y_context: Tensor, n_slots: int = N_SLOTS) -> tuple[Tensor, Tensor]:
    """TabICL wants classes numbered 0..k-1. Per table, the context labels renumbered in slot order, (b

In [4]:
z = prepare_table(x, n_context)
codes, slots = contiguous_codes(y_context)
print("standardised table (columns 0 and 1 agree; the missing cell is 0):"); print(z[0])
print("codes:", codes[0].tolist(), "| slot of each code:", slots[0, :2].tolist())
with torch.no_grad():
    cells = model.table.col_embedder._train_forward(z, codes, None, False)
print("cells", tuple(cells.shape), "(batch, rows, 4 CLS + columns, width)")

standardised table (columns 0 and 1 agree; the missing cell is 0):
tensor([[-1.41, -1.41,  0.17],
        [-0.71, -0.71, -1.18],
        [ 0.00,  0.00,  0.00],
        [ 0.71,  0.71,  1.52],
        [ 1.41,  1.41, -0.51],
        [-1.06, -1.06, -0.17],
        [ 1.06,  1.06, -0.85],
        [ 2.12,  2.12,  1.18]])
codes: [0, 1, 0, 1, 0] | slot of each code: [3, 5]
cells (1, 8, 7, 128) (batch, rows, 4 CLS + columns, width)


What the stage lets through: change one cell of test row 7 and see which rows' tokens move.

In [5]:
x_moved = x.clone(); x_moved[0, 7, 0] = 100.0
with torch.no_grad():
    cells_moved = model.table.col_embedder._train_forward(prepare_table(x_moved, n_context), codes, None, False)
print("rows whose tokens changed when test row 7, column 0 was moved:", changed(cells_moved, cells, dims=(-1, -2))[0].tolist())

rows whose tokens changed when test row 7, column 0 was moved: [0, 0, 0, 0, 0, 0, 0, 1]


## 3. The image side

With photos, every row carries one. Three steps turn it into tokens that join the row: a patch embedder, a small encoder over the patches of one photo, and the labelled read, where the patches of every photo read labels from the context photos.

The patch embedder is the one thing the encoder setting changes. Encoder `pixels` is ViT's linear patch embedding trained from scratch, one token per 14 x 14 square. The block encoders, `block4` (the chosen one), `block8` and `block12`, take the patch tokens of that block of a frozen DINOv2 ViT-B/14, computed outside the model, through a layer norm and one trained linear layer: 768 wide, 16 x 16 = 256 tokens for a 224 px photo. Encoder `cls` does the same with DINOv2's final CLS and register tokens.

In [6]:
from table_image_icl.model import PixelPatches, DinoPatches, ImageEncoder

show(PixelPatches, DinoPatches)

class PixelPatches(nn.Module):
    """One token per PATCH x PATCH square of pixels, from a linear layer trained from scratch."""

    def __init__(self, width: int):
        super().__init__()
        self.n_tokens = None                 # one token per patch: the count follows the image size
        self.project = nn.Conv2d(3, width, kernel_size=PATCH, stride=PATCH)

    def forward(self, images: Tensor) -> Tensor:
        """images (n, 3, H, W), normalised -> (n, patches, width), patches in row-major order."""
        return self.project(images).flatten(2).transpose(1, 2)

class DinoPatches(nn.Module):
    """Tokens of a frozen DINOv2 ViT-B/14 (computed outside the model, see inputs.py) through a layer norm and one
    trained linear layer. By default one token per 14 x 14 patch; with n_tokens, a fixed number of summary tokens
    instead (the final CLS and register tokens)."""

    def __init__(self, width: int, n_tokens: int | None = None):
        super().__init__()
        self.n

In [7]:
flat = images.flatten(0, 1)
with torch.no_grad():
    tokens = model.embed(flat)
    moved = flat.clone(); moved[0, :, :14, :14] += 1.0                  # touch the top-left patch of photo 0
    tokens_moved = model.embed(moved)
print("images", tuple(images.shape), "-> flattened", tuple(flat.shape), "-> patch tokens", tuple(tokens.shape), "(photos, patches, width)")
print("tokens of photo 0 changed by touching its top-left patch:", changed(tokens_moved[0], tokens[0]).tolist())
dino_input = torch.randn(x.shape[1], 256, 768)
print("encoder block4: input per photo", tuple(dino_input.shape), "->", tuple(DinoPatches(128)(dino_input).shape))

images (1, 8, 3, 28, 28) -> flattened (8, 3, 28, 28) -> patch tokens (8, 4, 128) (photos, patches, width)
tokens of photo 0 changed by touching its top-left patch: [1, 0, 0, 0]
encoder block4: input per photo (8, 256, 768) -> (8, 256, 128)


The encoder adds a learned position to each patch and runs a few transformer layers over the patches of one photo. No labels and no other photos are involved; photos in the batch do not see each other.

In [8]:
show(ImageEncoder.forward)
with torch.no_grad():
    encoded = model.encoder(tokens)
    encoded_moved = model.encoder(tokens_moved)
print("encoder", tuple(tokens.shape), "->", tuple(encoded.shape))
print("photos whose tokens changed when photo 0 was touched:", changed(encoded_moved, encoded, dims=(-1, -2)).tolist())

    def forward(self, tokens: Tensor) -> Tensor:
        """tokens (n, patches, width) -> (n, patches, width)"""
        tokens = tokens + self.position
        for layer in self.layers:
            tokens = layer(tokens)
        return tokens

encoder (8, 4, 128) -> (8, 4, 128)
photos whose tokens changed when photo 0 was touched: [1, 0, 0, 0, 0, 0, 0, 0]


The labelled read is this model's own. It is a cross-attention from every patch of every photo to the patches of the context photos: the query and the key are patch content, but the value is the label of the photo the key patch belongs to. So a test patch that looks like the patches of context photos labelled 5 reads back "5". A context photo is among its own sources, since the context is given. Relabelling one context photo changes what every photo reads back, while changing a test photo changes that photo only.

In [9]:
from table_image_icl.model import LabelRead, LabelledRead

show(LabelRead.forward, LabelledRead.forward)

    def forward(self, readers: Tensor, sources: Tensor, source_labels: Tensor, nearest: Tensor | None = None) -> Tensor:
        """readers (batch, photos, patches, width); sources (batch, S, patches, width); source_labels (batch, S, width).
        Without `nearest`, every patch attends to every source patch. With `nearest` (batch, photos, k), each photo
        attends only to the patches of its k neighbours. Returns what each reading patch read: the shape of readers."""
        batch, photos, patches, width = readers.shape
        queries, keys = self.norm_readers(readers), self.norm_sources(sources)
        if nearest is None:
            values = source_labels[:, :, None].expand(-1, -1, patches, -1).flatten(1, 2)
            read = self.attention(queries.flatten(1, 2), keys.flatten(1, 2), values)
            return read.reshape(batch, photos, patches, width)

        rows = torch.arange(batch, device=readers.device)[:, None, None]
        out = []
        for start in range(0, pho

In [10]:
encoded_rows = encoded.unflatten(0, (1, x.shape[1]))                   # back to (batch, rows, patches, width)
y_other = y_context.clone(); y_other[0, 1] = 3                       # relabel context photo 1
with torch.no_grad():
    read = model.read(encoded_rows, y_context)
    read_other = model.read(encoded_rows, y_other)
    test_moved = encoded_rows.clone(); test_moved[0, 7, 0, 0] += 1.0  # move one value of test photo 7
    read_moved = model.read(test_moved, y_context)
print("labelled read", tuple(encoded_rows.shape), "->", tuple(read.shape))
print("photos whose patches changed when context photo 1 was relabelled:", changed(read_other, read, dims=(-1, -2))[0].tolist())
print("photos whose patches changed when test photo 7 was changed:      ", changed(read_moved, read, dims=(-1, -2))[0].tolist())

labelled read (1, 8, 4, 128) -> (1, 8, 4, 128)
photos whose patches changed when context photo 1 was relabelled: [1, 1, 1, 1, 1, 1, 1, 1]
photos whose patches changed when test photo 7 was changed:       [0, 0, 0, 0, 0, 0, 0, 1]


## 4. The top-k read: retrieve, then read

The trained setting is `patch_read = "top_k"` with `patch_top_k = 32`: before reading, every photo picks the 32 context photos that look most like it and reads labels from their patches only. With 5 context photos, as above, that is every one of them, so the read so far was the plain one. Two steps, both in `table_image_icl/model.py`.

Retrieve: each photo is summarised by the mean of its patch tokens, the cosine similarity between its summary and every context photo's summary is taken, and the most similar context photos are kept. A nearest-neighbour lookup on content, with no labels involved. A context photo is its own nearest, so it reads its own label among others.

In [11]:
from table_image_icl.model import nearest_photos

show(nearest_photos)
with torch.no_grad():
    sources = encoded_rows[:, :n_context]
    summary = torch.nn.functional.normalize(encoded_rows.mean(2), dim=-1)
    similarity = summary @ torch.nn.functional.normalize(sources.mean(2), dim=-1).transpose(1, 2)
    nearest = nearest_photos(encoded_rows, sources, k=2)
print("cosine similarity of each photo's summary to each context photo's (photos x context):")
print(similarity[0])
print("the 2 nearest context photos of each photo:", nearest[0].tolist())
print("every context photo is among its own nearest:", all(i in nearest[0, i].tolist() for i in range(n_context)))

def nearest_photos(tokens: Tensor, sources: Tensor, k: int) -> Tensor:
    """For every photo, the k source photos whose mean patch token is most similar (cosine) to its own.
    tokens (batch, photos, patches, width), sources (batch, S, patches, width) -> (batch, photos, k) indices into S."""
    similarity = F.normalize(tokens.mean(2), dim=-1) @ F.normalize(sources.mean(2), dim=-1).transpose(1, 2)
    return similarity.topk(k, dim=-1).indices

cosine similarity of each photo's summary to each context photo's (photos x context):
tensor([[ 1.00,  0.05, -0.26,  0.23,  0.19],
        [ 0.05,  1.00,  0.28, -0.02,  0.26],
        [-0.26,  0.28,  1.00, -0.26,  0.09],
        [ 0.23, -0.02, -0.26,  1.00,  0.21],
        [ 0.19,  0.26,  0.09,  0.21,  1.00],
        [ 0.11, -0.01,  0.07,  0.16,  0.34],
        [-0.00,  0.05,  0.23, -0.09,  0.18],
        [-0.09,  0.26, -0.05,  0.22,  0.33]])
the 2 nearest context photos of each photo: [[0, 3], [1, 2], [2, 1], [3, 0], [4, 1], [4, 3], [2, 4], [4

Read: the `nearest` branch of `LabelRead.forward`, printed in section 3. For each photo, the patches of its neighbours are gathered as the keys and the neighbours' labels as the values, and the photo's patches attend over them. So a photo's query sees `top_k` x patches keys whatever the size of the context, which is why a model trained on at most 256 rows can be scored with 10,000 context rows unchanged: the attention problem it faces is the same.

The setting is an attribute, not a weight: the read built above is switched to 2 neighbours here. Relabelling a context photo now changes only the photos that retrieved it.

In [12]:
model.read.top_k = 2
far = next(i for i in range(n_context) if i not in nearest[0, 7].tolist())    # a context photo test photo 7 did not retrieve
near = nearest[0, 7, 0].item()                                                   # one it did
y_far, y_near = y_context.clone(), y_context.clone()
y_far[0, far], y_near[0, near] = 7, 7
with torch.no_grad():
    base = model.read(encoded_rows, y_context)
    read_far, read_near = model.read(encoded_rows, y_far), model.read(encoded_rows, y_near)
print("photos that retrieved context photo", far, ":", [i for i in range(x.shape[1]) if far in nearest[0, i].tolist()])
print("photos whose patches changed when it was relabelled:  ", changed(read_far, base, dims=(-1, -2))[0].tolist())
print("photos that retrieved context photo", near, ":", [i for i in range(x.shape[1]) if near in nearest[0, i].tolist()])
print("photos whose patches changed when it was relabelled:  ", changed(read_near, base, dims=(-1, -2))[0].tolist())
model.read.top_k = 32                                                            # back to the trained setting

photos that retrieved context photo 0 : [0, 3]
photos whose patches changed when it was relabelled:   [1, 0, 0, 1, 0, 0, 0, 0]
photos that retrieved context photo 4 : [4, 5, 6, 7]
photos whose patches changed when it was relabelled:   [0, 0, 0, 0, 1, 1, 1, 1]


## 5. The row stage

`TableImageICL.photo_tokens` strings the three image steps together and adds a learned marker that tells the row stage a token is a patch and not a cell. The row stage is TabICL's: a row's tokens, its 4 CLS tokens, its cells and now its patches, attend to each other, and the CLS tokens are read out and concatenated, so a row becomes one vector of width 4 x 128. Each row is handled on its own.

In [13]:
show(TableImageICL.photo_tokens)
with torch.no_grad():
    patches = model.photo_tokens(images, y_context)
    rows = model.table.row_interactor._train_forward(torch.cat([cells, patches], 2), None)
    images_moved = images.clone(); images_moved[0, 7, :, :14, :14] += 1.0                     # touch test row 7's photo
    rows_moved = model.table.row_interactor._train_forward(torch.cat([cells, model.photo_tokens(images_moved, y_context)], 2), None)
print("photo tokens", tuple(patches.shape), "-> rows with photos", tuple(rows.shape),
      "| tokens per row: 4 CLS +", x.shape[2], "cells +", patches.shape[2], "patches")
print("rows changed by touching test row 7's photo:", changed(rows_moved, rows)[0].tolist())

    def photo_tokens(self, images: Tensor, y_context: Tensor) -> Tensor:
        """images: (batch, rows, 3, H, W) for encoder pixels, (batch, rows, tokens, DINO_WIDTH) for the DINOv2 encoders.
        Returns each row's patch tokens after the encoder and the labelled read: (batch, rows, patches, width)."""
        batch, rows = images.shape[:2]
        tokens = self.encoder(self.embed(images.flatten(0, 1))).unflatten(0, (batch, rows))
        return self.read(tokens, y_context) + self.patch_marker

photo tokens (1, 8, 4, 128) -> rows with photos (1, 8, 512) | tokens per row: 4 CLS + 3 cells + 4 patches
rows changed by touching test row 7's photo: [0, 0, 0, 0, 0, 0, 0, 1]


## 6. The in-context stage and the output

Now rows attend to rows. The context rows get their label added; then every row attends to the context rows, and the test rows' vectors are scored over the k classes of the task. A test row reads which labelled rows it resembles; because the keys are the context rows, a test row never influences another row. The scores over the codes are put back on the task's slots, and slots absent from the context get no probability. That is `TableImageICL.forward`: the stages above in sequence.

In [14]:
show(TableImageICL.forward)
with torch.no_grad():
    logits = model.table.icl_predictor._icl_predictions(rows, codes)[:, n_context:]
    log_probs = model(x, y_context, images=images)
    rows_test_moved = rows.clone(); rows_test_moved[0, 6, 0] += 1.0                           # move one value of test row 6
    rows_context_moved = rows.clone(); rows_context_moved[0, 0, 0] += 1.0                     # move one value of context row 0
    out = model.table.icl_predictor._icl_predictions(rows, codes)
    out_test_moved = model.table.icl_predictor._icl_predictions(rows_test_moved, codes)
    out_context_moved = model.table.icl_predictor._icl_predictions(rows_context_moved, codes)
print("in-context stage", tuple(rows.shape), "-> logits over the codes", tuple(logits.shape))
print("rows whose scores changed by changing test row 6:   ", changed(out_test_moved, out)[0].tolist())
print("rows whose scores changed by changing context row 0:", changed(out_context_moved, out)[0].tolist())
print("forward", tuple(log_probs.shape), "(batch, test rows, label slots); probabilities per test row, slots 3 and 5 only:")
print(log_probs[0].exp())
print("largest difference from the stage-by-stage logits:", f"{(log_probs[0][:, [3, 5]] - logits[0][:, :2].log_softmax(-1)).abs().max():.1e}")

    def forward(self, x: Tensor, y_context: Tensor, images: Tensor | None = None) -> Tensor:
        """x (batch, rows, columns), context rows first, NaN = missing; y_context (batch, n_context) label slots.
        Returns log-probabilities over the label slots for the test rows: (batch, rows - n_context, N_SLOTS)."""
        n_context = y_context.shape[1]
        codes, slots = contiguous_codes(y_context)
        tokens = self.table.col_embedder._train_forward(prepare_table(x, n_context), codes, None, False)   # (batch, rows, cls + columns, width)
        if images is not None:
            tokens = torch.cat([tokens, self.photo_tokens(images, y_context)], 2)
        rows = self.table.row_interactor._train_forward(tokens, None)
        logits = self.table.icl_predictor._icl_predictions(rows, codes)[:, n_context:]                    # over the codes
        out = logits.new_full((len(x), logits.shape[1], N_SLOTS), float("-inf"))
        for b in range(len(x)):
            k = int((slots

in-context stage (1, 8, 512) -> logits over the codes (1, 3, 10)
rows whose scores changed by changing test row 6:    [0, 0, 0, 0, 0, 0, 1, 0]
rows whose scores changed by changing context row 0: [1, 1, 1, 1, 1, 1, 1, 1]
forward (1, 3, 10) (batch, test rows, label slots); probabilities per test row, slots 3 and 5 only:


tensor([[0.00, 0.00, 0.00, 0.76, 0.00, 0.24, 0.00, 0.00, 0.00, 0.00],
        [0.00, 0.00, 0.00, 0.51, 0.00, 0.49, 0.00, 0.00, 0.00, 0.00],
        [0.00, 0.00, 0.00, 0.36, 0.00, 0.64, 0.00, 0.00, 0.00, 0.00]])
largest difference from the stage-by-stage logits: 0.0e+00


The training loss is the negative log-likelihood of the true slot over the test rows whose slot appears in the context; a test row whose class the context never shows is left out, since nothing in the task could predict it.

In [15]:
from table_image_icl.training import task_loss

show(task_loss)
loss, accuracy = task_loss(log_probs, y_context, y_test)
print(f"loss {loss:.3f} (chance for two classes is log 2 = 0.693), accuracy {accuracy:.2f}")
y_unseen = torch.tensor([[3, 5, 7]])                                   # slot 7 is in no context row
loss_unseen, _ = task_loss(log_probs, y_context, y_unseen)
print(f"with test row 7 labelled with an unseen slot, the loss is over two rows only: {loss_unseen:.3f}")

def task_loss(log_probs: Tensor, y_context: Tensor, y_test: Tensor) -> tuple[Tensor, float]:
    """Mean negative log-likelihood and accuracy over the test rows whose class appears in their task's context;
    a class the context never shows cannot be predicted from it."""
    predictable = (y_test[:, :, None] == y_context[:, None, :]).any(-1)
    loss = F.nll_loss(log_probs[predictable], y_test[predictable])
    accuracy = (log_probs.argmax(-1) == y_test)[predictable].float().mean().item()
    return loss, accuracy

loss 0.481 (chance for two classes is log 2 = 0.693), accuracy 0.67
with test row 7 labelled with an unseen slot, the loss is over two rows only: 0.498
